# Load & Export Blackrock NEV/NSx Data

Files are picked by **role prefix** :
- `NSP-*.nev` — behavioral comments  → `Blackrock_YYYY-MM-DD_expmeta.txt`, `Blackrock_YYYY-MM-DD_trials.csv`
- `NSP-*.ns2` — analog / eye data    → `Blackrock_YYYY-MM-DD_analog.mat` (segmented per trial, `load_analog=True`)
- `HUB-*.nev` — online spike timing  → `Blackrock_YYYY-MM-DD_spikes.mat` (rasterized per trial, `load_online_spikes=True`)
- `HUB-*.nev` — online spike waveforms → `Blackrock_YYYY-MM-DD_waveforms.mat` (segmented per trial, `load_online_wave=True`)

`HUB-*.nev` is also the legacy fallback for comments. For analog, set `ns_marker='ns6'` etc. for other NSx types.

By default the spike/waveform outputs keep **only sorted units** — unsorted threshold crossings (unit 0) and noise (unit 255) are dropped. Set `include_unsorted=True` to retain every unit (much larger waveform output).

The loading can be applied on single folders or multiple folders in batch.

In [1]:
from pathlib import Path
from jlab import BlackRockLoader

## 1. Set paths

`Session_Path` (Basic_Path / Monkey / Location) is used by **both** parts. 

`Date` : single date for individual loading, <br>
         a string of sates {'2026-06-17','2026-06-18'} for batch loading, <br>
         {} for loading all year-month-date files in the path <br>

In [2]:
# ── Edit these variables ───────────────────────────────────────────────────
Basic_Path = '/Users/xuefeiyu/Documents/XuefeiFile/WorkRelated/Data'
Monkey     = 'Monkey test'
#Monkey    = 'Monkey Athos'
Location   = 'in_lab'
Date       = '2026-06-24'     # year-month-date folder; also used as the date string
# Dates = ['2026-06-17', '2026-06-18']   # a list of YYYY-MM-DD folders for multiple loadings, use .batch_loading()
# Dates = [] #load and export data for all dates in the specified path, use .batch_loading()
# ─────────────────────────────────────────────────────────────────────────

# Session directory: the folder that contains raw_data/ and export_data/.
# The .nev (and optional .ns2) files are auto-detected from Session_Path/raw_data/Date.
Session_Path = Path(Basic_Path) / Monkey / Location
print('Session path:', Session_Path)
print('Date        :', Date)

Session path: /Users/xuefeiyu/Documents/XuefeiFile/WorkRelated/Data/Monkey test/in_lab
Date        : 2026-06-24


## 1. Load NEV and check comments

In [3]:
# Peek at the .nev files in this session folder (largest first):
#BlackRockLoader.list_nev_files(Session_Path, Date) #Optional : to list all the nev files for checking 

loader = BlackRockLoader(
    session_path = Session_Path, 
    date = Date,
    load_analog=True,        # segment the NSP-*.ns2 analog (eye-tracking) file
    load_online_spikes=True, # rasterize online spikes from the HUB-*.nev file
    load_online_wave=True,   # also segment per-spike waveforms (µV) from the HUB-*.nev file
    include_unsorted=False,  # default false: drop unsorted (unit 0) + noise (255); set True to keep all units
    # nev_filename='...',    # optional: pick a specific comment .nev (else largest NSP-*)
    # ns_marker='ns2',       # NSx extension to load; default 'ns2', e.g. 'ns6'
    # ns_filename='...',     # optional: pick a specific NSx file
    # pre_ms=500, post_ms=500, bin_ms=1,   # segmentation buffers / raster bin width (ms)
)


# loader.print_comments() # check comments

#If load multiple dates, use the run_batch method instead.
"""
Dates = ['2026-06-17', '2026-06-18']   # a list of YYYY-MM-DD folders
# Dates = None                          # or: None -> every date folder under raw_data/

report = BlackRockLoader.run_batch(
    Session_Path,
    dates=Dates,
    load_analog=True,          # also segment each day's NSP-*.ns2 analog file
    load_online_spikes=True,   # also rasterize each day's HUB-*.nev online spikes
    load_online_wave=True,     # also segment each day's HUB-*.nev spike waveforms
    include_unsorted=False,    # default: drop unsorted (unit 0) + noise (255); set True to keep all units
    skip_existing=False,      # skip dates already exported (re-run only does new ones); default False
    # ns_marker='ns2',       # NSx extension; default 'ns2'
)
"""


Comment NEV   : NSP-06242026_test_for_timingissue.nev, Hub1-06242026_test_for_timingissue.nev
Analog file   : NSP-06242026_test_for_timingissue.ns2
Spike NEV     : Hub1-06242026_test_for_timingissue.nev (spikes+waveforms)
Output dir    : /Users/xuefeiyu/Documents/XuefeiFile/WorkRelated/Data/Monkey test/in_lab/export_data/2026-06-24


"\nDates = ['2026-06-17', '2026-06-18']   # a list of YYYY-MM-DD folders\n# Dates = None                          # or: None -> every date folder under raw_data/\n\nreport = BlackRockLoader.run_batch(\n    Session_Path,\n    dates=Dates,\n    load_analog=True,          # also segment each day's NSP-*.ns2 analog file\n    load_online_spikes=True,   # also rasterize each day's HUB-*.nev online spikes\n    load_online_wave=True,     # also segment each day's HUB-*.nev spike waveforms\n    include_unsorted=False,    # default: drop unsorted (unit 0) + noise (255); set True to keep all units\n    skip_existing=False,      # skip dates already exported (re-run only does new ones); default False\n    # ns_marker='ns2',       # NSx extension; default 'ns2'\n)\n"

# 2. Export the data

In [4]:
# export the data
output_dir, files = loader.run()

print(f"\nOutput folder: {output_dir}")
for f in files:
    print(f"  {f}")


NSP-06242026_test_for_timingissue.nev opened

Hub1-06242026_test_for_timingissue.nev opened
Loading NEV: Hub1-06242026_test_for_timingissue.nev  (3100 events)
  86 trials parsed (86 complete)
Loading analog: NSP-06242026_test_for_timingissue.ns2

NSP-06242026_test_for_timingissue.ns2 opened

NSP-06242026_test_for_timingissue.ns2 closed
  3 channels, 351921 samples, 1000.0 Hz
  Analog segmented (86 trials) -> Blackrock_2026-06-24_analog.mat
Loading spikes: Hub1-06242026_test_for_timingissue.nev

Hub1-06242026_test_for_timingissue.nev opened
  0 spikes loaded (dropped 25223726 unsorted/noise) (waveforms 48 samples/spike)
  No sorted spikes found; skipping spike/waveform export.

Output folder: /Users/xuefeiyu/Documents/XuefeiFile/WorkRelated/Data/Monkey test/in_lab/export_data/2026-06-24
  Blackrock_2026-06-24_expmeta.txt
  Blackrock_2026-06-24_trials.csv
  Blackrock_2026-06-24_analog.mat


# 3Check the trial summary

In [5]:
loader.print_summary() 

Session Summary
──────────────────────────────────────────────────
Sessions       : 2
Total length   : 5 min 28.3 sec
Total trials   : 86  (86 complete)

Task: visual_saccades_experiment
  Total          : 59
  Correct        : 56
  Outcomes:
    correct                        : 56
    broke_fixation                 : 2
    timeout                        : 1

Task: time_delay_experiment [choice]
  Total          : 27
  Successful     : 27  (correct + error)
  Outcomes:
    correct                        : 21
    wrong                          : 6


## 4. If batch loading, check the batch report

`run_batch` already prints a per-date `... OK / SKIPPED / FAILED` line and a final `Done:` tally. The cell below inspects the returned `report` programmatically — list every date's status and surface the error message for any date that failed, so problems are easy to spot.

In [ ]:
"""
Dates = ['2026-06-17', '2026-06-18']   # a list of YYYY-MM-DD folders
# Dates = None                          # or: None -> every date folder under raw_data/

report = BlackRockLoader.run_batch(
    Session_Path,
    dates=Dates,
    load_analog=True,          # also segment each day's NSP-*.ns2 analog file
    load_online_spikes=True,   # also rasterize each day's HUB-*.nev online spikes
    load_online_wave=True,     # also segment each day's HUB-*.nev spike waveforms
    include_unsorted=False,    # default: drop unsorted (unit 0) + noise (255); set True to keep all units
    skip_existing=False,      # skip dates already exported (re-run only does new ones); default False
    # ns_marker='ns2',       # NSx extension; default 'ns2'
)
"""

"""
# Per-date status
for r in report:
    print(f"{r['date']}  {r['status']}")

# Surface any failures
failed = [r for r in report if r['status'] == 'failed']
if failed:
    print("\nFailed dates:")
    for r in failed:
        print(f"  {r['date']}: {r['error']}")
else:
    print("\nAll requested dates processed (none failed).")
"""